<a href="https://colab.research.google.com/github/kavyasudha12/Gen-AI-Training-Task/blob/main/week2_day1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# 1. Install dependencies
!pip install -q sentence-transformers faiss-cpu


# 2. Imports
import faiss
import numpy as np
from sentence_transformers import SentenceTransformer


# 3. Dataset
dataset = """
Artificial intelligence (AI) refers to the simulation of human intelligence in machines.
AI systems are capable of learning, reasoning, problem-solving, perception, and language understanding.
Machine learning is a subset of AI that focuses on learning from data.
Supervised learning requires labeled datasets for training models.
Unsupervised learning discovers hidden patterns in unlabeled data.
Deep learning uses neural networks with multiple layers.
Large language models are a type of deep learning model trained on vast text corpora.
These models can perform tasks such as translation, summarization, and question answering.
Transformer architecture revolutionized natural language processing.
Attention mechanisms allow models to focus on relevant input tokens.
Embeddings are numerical representations of text or data.
Embeddings capture semantic meaning in vector space.
Vector databases store embeddings efficiently.
Similarity search retrieves vectors closest to a query vector.
FAISS is a library developed by Facebook AI Research for efficient similarity search.
FAISS supports clustering, indexing, and fast nearest neighbor search.
Chroma is a modern open-source vector database for AI applications.
Retrieval Augmented Generation (RAG) combines information retrieval with text generation.
RAG improves factual accuracy of language models.
"""


# 4. CORRECT CHUNKING (Sentence-based)
def chunk_text_by_sentences(text):
    sentences = [s.strip() for s in text.split("\n") if s.strip()]
    return sentences


chunks = chunk_text_by_sentences(dataset)

print("Total chunks created:", len(chunks))


# 5. Load embedding model
model = SentenceTransformer("all-MiniLM-L6-v2")

# 6. Create embeddings
embeddings = model.encode(chunks)
embeddings = np.array(embeddings).astype("float32")

print("Embeddings shape:", embeddings.shape)


# 7. Store in FAISS
dimension = embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(embeddings)

print("Vectors stored in FAISS:", index.ntotal)

# 8. Retrieval function
def retrieve(query, top_k=3):
    top_k = min(top_k, index.ntotal)

    query_embedding = model.encode([query])
    query_embedding = np.array(query_embedding).astype("float32")

    distances, indices = index.search(query_embedding, top_k)

    return [chunks[i] for i in indices[0]]


# 9. Test queries
queries = [
    "What is FAISS?",
    "Explain embeddings",
    "What is Retrieval Augmented Generation?"
]

for q in queries:
    print(f"\nQuery: {q}")
    results = retrieve(q, top_k=3)
    for i, res in enumerate(results, 1):
        print(f"Result {i}: {res}")
